# DMET-SSTKG — S-STKG Construction Protocol: AutoLaparo External Source-Counterpart Stress-Test Cohort

This notebook constructs phase-aligned S-STKGs from AutoLaparo videos and native frame-phase annotations. Native phase IDs and non-monotonic transitions are preserved; projection to the predefined source-counterpart space occurs only during external evaluation.


Each example call processes one surgical case. To construct S-STKGs for
additional cases, repeat the relevant call with the corresponding local input
paths and assign a unique `manual_case_id` to each case. The number of cases
processed is determined by the user and is not fixed by this notebook.

## 1. Required Environment

The recorded construction environment used Python 3.9.19, Ultralytics 8.4.18,
and PyTorch 2.4.1.

Install the notebook packages before running:

```bash
pip install ultralytics==8.4.18 opencv-python pandas numpy
```

Install a PyTorch 2.4.1 build compatible with the local CPU or CUDA
environment. The original surgical datasets and the trained node-detector
checkpoint are not distributed with this notebook and must be obtained
separately.

In [ ]:
from pathlib import Path
import re

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

In [ ]:
VIRTUAL_CONTEXT_CLASS_ID = 8


def require_path(path, description):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"{description} was not found: {path}"
        )

## 2. Native Phase Representation and Phase-Aligned Fragment Generation

AutoLaparo annotations provide one native phase observation per second. Annotation times are converted to video-frame boundaries using the source video frame rate. A new segment starts whenever the observed native phase changes, preserving non-monotonic transitions.

In [ ]:
AUTOLAPARO_PHASE_ID_TO_NAME = {
    1: "Preparation",
    2: "DividingLigamentAndPeritoneum",
    3: "DividingUterineVesselsAndLigament",
    4: "TransectingTheVagina",
    5: "SpecimenRemoval",
    6: "Suturing",
    7: "Washing",
}


def read_autolaparo_frame_phase_labels(
    label_path,
):
    rows = []

    with open(
        label_path,
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as label_file:
        for line_number, raw_line in enumerate(
            label_file,
            start=1,
        ):
            line = raw_line.strip()

            if not line:
                continue

            parts = re.split(
                r"[\s,\t]+",
                line,
            )

            if (
                len(parts) < 2
                or not parts[0].isdigit()
            ):
                continue

            frame_value = int(
                parts[0]
            )
            phase_value = int(
                parts[1]
            )

            if phase_value not in (
                AUTOLAPARO_PHASE_ID_TO_NAME
            ):
                raise ValueError(
                    f"Invalid AutoLaparo phase "
                    f"{phase_value} at line "
                    f"{line_number}"
                )

            rows.append(
                {
                    "Frame": frame_value,
                    "Phase": phase_value,
                    "PhaseName": (
                        AUTOLAPARO_PHASE_ID_TO_NAME[
                            phase_value
                        ]
                    ),
                }
            )

    if not rows:
        raise RuntimeError(
            f"No usable phase labels were found in {label_path}"
        )

    frame_table = pd.DataFrame(
        rows
    )

    if frame_table["Frame"].min() == 1:
        frame_table["second_index"] = (
            frame_table["Frame"] - 1
        )
    else:
        frame_table["second_index"] = (
            frame_table["Frame"]
        )

    return frame_table


def split_autolaparo_video_by_phases(
    video_path,
    label_path,
    output_base_dir,
    manual_case_id,
):
    video_path = Path(video_path)
    label_path = Path(label_path)

    if not video_path.exists():
        raise FileNotFoundError(
            f"Input video was not found: {video_path}"
        )

    if not label_path.exists():
        raise FileNotFoundError(
            f"Phase-label file was not found: {label_path}"
        )

    case_folder = (
        Path(output_base_dir)
        / f"Case_{manual_case_id:02d}"
    )
    case_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    frame_table = (
        read_autolaparo_frame_phase_labels(
            label_path
        )
    )

    capture = cv2.VideoCapture(
        str(video_path)
    )

    if not capture.isOpened():
        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    fps = float(
        capture.get(
            cv2.CAP_PROP_FPS
        )
    )
    width = int(
        capture.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )
    height = int(
        capture.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )
    total_frames = int(
        capture.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if fps <= 0 or total_frames <= 0:
        capture.release()
        raise RuntimeError(
            f"Invalid video metadata: {video_path}"
        )

    frame_table["phase_change"] = (
        frame_table["Phase"]
        != frame_table["Phase"].shift(1)
    )
    segment_starts = (
        frame_table.index[
            frame_table["phase_change"]
        ].tolist()
    )
    segment_ends = (
        segment_starts[1:]
        + [len(frame_table)]
    )

    written_segments = 0

    for start_row, end_row in zip(
        segment_starts,
        segment_ends,
    ):
        phase_id = int(
            frame_table.iloc[
                start_row
            ]["Phase"]
        )
        phase_name = str(
            frame_table.iloc[
                start_row
            ]["PhaseName"]
        )

        start_second = int(
            frame_table.iloc[
                start_row
            ]["second_index"]
        )
        end_second_exclusive = (
            int(
                frame_table.iloc[
                    end_row - 1
                ]["second_index"]
            )
            + 1
        )

        start_frame = int(
            round(
                start_second * fps
            )
        )
        end_frame = int(
            round(
                end_second_exclusive
                * fps
            )
        )

        start_frame = max(
            0,
            min(
                start_frame,
                total_frames,
            ),
        )
        end_frame = max(
            start_frame + 1,
            min(
                end_frame,
                total_frames,
            ),
        )

        base_name = (
            f"seg_{written_segments:02d}_"
            f"{phase_name}_id{phase_id}"
        )

        video_output = (
            case_folder
            / f"{base_name}.mp4"
        )
        label_output = (
            case_folder
            / f"{base_name}.txt"
        )

        writer = cv2.VideoWriter(
            str(video_output),
            cv2.VideoWriter_fourcc(
                *"mp4v"
            ),
            fps,
            (width, height),
        )

        capture.set(
            cv2.CAP_PROP_POS_FRAMES,
            start_frame,
        )

        for frame_index in range(
            start_frame,
            end_frame,
        ):
            success, frame = capture.read()

            if not success:
                break

            writer.write(frame)

        writer.release()

        with label_output.open(
            "w",
            encoding="utf-8",
        ) as label_file:
            label_file.write(
                "frame_idx\tphase_id\n"
            )

            for frame_index in range(
                start_frame,
                end_frame,
            ):
                label_file.write(
                    f"{frame_index}\t{phase_id}\n"
                )

        print(
            f"  {phase_name} | native phase "
            f"{phase_id} | frame "
            f"{start_frame}-{end_frame}"
        )

        written_segments += 1

    capture.release()

    print(
        f"Saved {written_segments} segment(s) "
        f"to {case_folder}"
    )

### Example call: AutoLaparo phase fragmentation

Replace the example paths before running:

- `video_path`: one full AutoLaparo video.
- `label_path`: its matching frame-phase label file.
- `output_base_dir`: directory where `Case_XX` fragment folders will be written.
- `manual_case_id`: unique integer assigned to this case.



Copy this call cell and replace its paths and `manual_case_id` to process
another case.

In [ ]:
split_autolaparo_video_by_phases(
    video_path="/path/to/AutoLaparo/videos/01.mp4",
    label_path="/path/to/AutoLaparo/labels/label_01.txt",
    output_base_dir="/path/to/AutoLaparo_Video_Segments",
    manual_case_id=1,
)

## 3. S-STKG Node Generation

The fragment videos retain the source video frame rate, so the detector is applied to every fifth frame.

The detector's physical class ID is written directly to the node file without
a mapping table or runtime remapping. One virtual-context node is added
programmatically to every sampled frame with `class=8`, `isV=1`, zero geometry,
and zero motion.

For this cohort, use `sampling_rate=5`.

### Path fields

- `model_path`: trained node-detector checkpoint.
- `case_path`: one phase-fragment `Case_XX` directory created above.
- `output_root`: root directory where the S-STKG `Case_XX` directory will be
  created.

In [ ]:
class STKGGenerator:
    def __init__(
        self,
        model_path,
        conf=0.25,
        imgsz=640,
    ):
        model_path = Path(model_path)
        require_path(
            model_path,
            "Node-detector checkpoint",
        )

        self.model = YOLO(
            str(model_path)
        )
        self.conf = float(conf)
        self.imgsz = int(imgsz)
        self.previous_centers = {}

    def process_single_case(
        self,
        case_path,
        output_root,
        sampling_rate=5,
    ):
        case_path = Path(case_path)
        output_root = Path(output_root)
        sampling_rate = int(
            sampling_rate
        )

        if sampling_rate <= 0:
            raise ValueError(
                "sampling_rate must be "
                "a positive integer"
            )

        if not case_path.is_dir():
            raise FileNotFoundError(
                "Phase-fragment case "
                f"directory was not found: "
                f"{case_path}"
            )

        output_case_directory = (
            output_root
            / case_path.name
        )
        output_case_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        video_files = sorted(
            case_path.glob("*.mp4")
        )

        if not video_files:
            raise RuntimeError(
                f"No fragment videos were "
                f"found in {case_path}"
            )

        global_max_frame = 1

        for video_path in video_files:
            label_path = (
                video_path.with_suffix(".txt")
            )

            if not label_path.exists():
                continue

            labels = pd.read_csv(
                label_path,
                sep="\t",
            )
            labels.columns = [
                str(column).strip()
                for column in labels.columns
            ]

            if (
                "frame_idx" not in labels.columns
                or "phase_id"
                not in labels.columns
            ):
                raise ValueError(
                    "Expected frame_idx and "
                    "phase_id columns in "
                    f"{label_path}"
                )

            if not labels.empty:
                global_max_frame = max(
                    global_max_frame,
                    int(
                        labels[
                            "frame_idx"
                        ].max()
                    ),
                )

        print(
            "Generating S-STKG nodes for "
            f"{case_path.name}"
        )

        for video_path in video_files:
            label_path = (
                video_path.with_suffix(".txt")
            )

            if not label_path.exists():
                print(
                    "Missing label file; "
                    f"skipped: {video_path.name}"
                )
                continue

            self.previous_centers = {}

            self.generate_stkg(
                sub_video_path=video_path,
                sub_label_path=label_path,
                output_directory=(
                    output_case_directory
                ),
                sampling_rate=(
                    sampling_rate
                ),
                global_total_frames=(
                    global_max_frame
                ),
            )

    def generate_stkg(
        self,
        sub_video_path,
        sub_label_path,
        output_directory,
        sampling_rate,
        global_total_frames,
    ):
        sub_video_path = Path(
            sub_video_path
        )
        sub_label_path = Path(
            sub_label_path
        )
        output_directory = Path(
            output_directory
        )

        case_match = re.search(
            r"Case_(\d+)",
            str(sub_video_path),
        )
        case_id = (
            int(case_match.group(1))
            if case_match
            else 0
        )

        capture = cv2.VideoCapture(
            str(sub_video_path)
        )

        if not capture.isOpened():
            raise RuntimeError(
                f"Could not open fragment: "
                f"{sub_video_path}"
            )

        phase_total_frames = int(
            capture.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
        )

        if phase_total_frames <= 0:
            capture.release()
            raise RuntimeError(
                f"Invalid fragment length: "
                f"{sub_video_path}"
            )

        labels = pd.read_csv(
            sub_label_path,
            sep="\t",
        )
        labels.columns = [
            str(column).strip()
            for column in labels.columns
        ]

        required_label_columns = {
            "frame_idx",
            "phase_id",
        }
        missing_label_columns = (
            required_label_columns.difference(
                labels.columns
            )
        )

        if missing_label_columns:
            capture.release()
            raise ValueError(
                f"{sub_label_path} is missing "
                f"columns: "
                f"{sorted(missing_label_columns)}"
            )

        node_path = (
            output_directory
            / (
                f"{sub_video_path.stem}"
                "_nodes.txt"
            )
        )
        output_label_path = (
            output_directory
            / (
                f"{sub_video_path.stem}"
                "_labels.txt"
            )
        )

        sampled_frames = 0
        physical_nodes = 0
        virtual_nodes = 0
        no_detection_frames = 0
        detector_class_counts = {}

        with node_path.open(
            "w",
            encoding="utf-8",
        ) as node_file, output_label_path.open(
            "w",
            encoding="utf-8",
        ) as output_label_file:
            node_file.write(
                "x,y,w,h,vx,vy,class,"
                "case_id,g_ratio,p_ratio,isV\n"
            )
            output_label_file.write(
                "frame_idx,phase_id\n"
            )

            phase_frame_index = 0

            while capture.isOpened():
                success, frame = (
                    capture.read()
                )

                if not success:
                    break

                if (
                    phase_frame_index
                    % sampling_rate
                    == 0
                ):
                    if (
                        phase_frame_index
                        >= len(labels)
                    ):
                        break

                    row = labels.iloc[
                        phase_frame_index
                    ]

                    global_frame_index = int(
                        row["frame_idx"]
                    )
                    phase_id = int(
                        row["phase_id"]
                    )

                    global_progress = (
                        global_frame_index
                        / max(
                            global_total_frames,
                            1,
                        )
                    )
                    phase_progress = (
                        phase_frame_index
                        / max(
                            phase_total_frames,
                            1,
                        )
                    )

                    result = self.model(
                        frame,
                        conf=self.conf,
                        imgsz=self.imgsz,
                        verbose=False,
                    )[0]
                    boxes = result.boxes

                    if (
                        boxes is None
                        or len(boxes) == 0
                    ):
                        no_detection_frames += 1

                    if boxes is not None:
                        for box in boxes:
                            x, y, width, height = (
                                box.xywhn[0]
                                .tolist()
                            )
                            class_id = int(
                                box.cls[0]
                            )

                            previous_x, previous_y = (
                                self.previous_centers.get(
                                    class_id,
                                    (x, y),
                                )
                            )
                            velocity_x = (
                                x - previous_x
                            )
                            velocity_y = (
                                y - previous_y
                            )
                            self.previous_centers[
                                class_id
                            ] = (x, y)

                            node_file.write(
                                f"{x:.4f},{y:.4f},"
                                f"{width:.4f},"
                                f"{height:.4f},"
                                f"{velocity_x:.4f},"
                                f"{velocity_y:.4f},"
                                f"{class_id},"
                                f"{case_id},"
                                f"{global_progress:.4f},"
                                f"{phase_progress:.4f},0\n"
                            )

                            physical_nodes += 1
                            detector_class_counts[
                                class_id
                            ] = (
                                detector_class_counts.get(
                                    class_id,
                                    0,
                                )
                                + 1
                            )

                    # The virtual-context node is
                    # programmatic, not a detector output.
                    node_file.write(
                        "0,0,0,0,0,0,"
                        f"{VIRTUAL_CONTEXT_CLASS_ID},"
                        f"{case_id},"
                        f"{global_progress:.4f},"
                        f"{phase_progress:.4f},1\n"
                    )
                    virtual_nodes += 1

                    output_label_file.write(
                        f"{global_frame_index},"
                        f"{phase_id}\n"
                    )
                    sampled_frames += 1

                phase_frame_index += 1

        capture.release()

        print(
            f"  {sub_video_path.stem} | "
            f"sampled_frames="
            f"{sampled_frames} | "
            f"physical_nodes="
            f"{physical_nodes} | "
            f"virtual_nodes="
            f"{virtual_nodes} | "
            f"no_detection_frames="
            f"{no_detection_frames}"
        )
        print(
            "    detector_class_counts="
            + str(
                dict(
                    sorted(
                        detector_class_counts.items()
                    )
                )
            )
        )

In [ ]:
model_path = "/path/to/node_detector/weights/best.pt"
case_path = "/path/to/AutoLaparo_Video_Segments/Case_01"
output_root = "/path/to/AutoLaparo_STKG"

generator = STKGGenerator(
    model_path=model_path,
    conf=0.25,
    imgsz=640,
)
generator.process_single_case(
    case_path=case_path,
    output_root=output_root,
    sampling_rate=5,
)

## 4. Physical and Virtual Edge Generation

The edge threshold was not re-optimized for AutoLaparo. Although AutoLaparo
represents a different surgical procedure and native phase ontology, its
S-STKGs use the same normalized bounding-box representation,
every-fifth-frame node sampling, and scale-normalized physical-edge criterion
defined in the source construction. To preserve a fixed, target-independent
construction protocol and avoid external-cohort-specific tuning, the
source-selected threshold was applied unchanged.

In the source-side audit, thresholds from 0.10 to 0.90 were evaluated in
increments of 0.10. The value 0.30 was retained because it was the first
operating point showing the maximum observed marginal increase in mean
physical connectivity. Higher thresholds primarily produced progressively
denser graphs without a comparable additional connectivity gain. This
selection was completed before the AutoLaparo external evaluation and does
not imply that 0.30 was independently optimized for this dataset.

Using the fixed threshold of 0.30, physical edges follow the scale-normalized
proximity rule and are stored bidirectionally. Every physical node is also
connected bidirectionally to the virtual-context node.

In [ ]:
class SurgicalEdgeGenerator:
    def __init__(
        self,
        optimal_tau=0.30,
    ):
        self.tau = float(
            optimal_tau
        )

        if self.tau <= 0:
            raise ValueError(
                "optimal_tau must be positive"
            )

    def process_all_files(
        self,
        folder_path,
    ):
        folder_path = Path(
            folder_path
        )

        if not folder_path.is_dir():
            raise FileNotFoundError(
                f"S-STKG case directory "
                f"was not found: "
                f"{folder_path}"
            )

        node_files = sorted(
            folder_path.glob(
                "*_nodes.txt"
            )
        )

        if not node_files:
            raise RuntimeError(
                f"No node files were found "
                f"in {folder_path}"
            )

        total_edges = 0

        for node_path in node_files:
            edge_path = (
                folder_path
                / node_path.name.replace(
                    "_nodes.txt",
                    "_edges.txt",
                )
            )

            edge_count = (
                self.generate_edges(
                    nodes_file_path=(
                        node_path
                    ),
                    output_edges_path=(
                        edge_path
                    ),
                )
            )
            total_edges += edge_count

            print(
                f"  {node_path.name} -> "
                f"{edge_path.name} | "
                f"edges={edge_count}"
            )

        print(
            f"Total edges: {total_edges}"
        )

    def generate_edges(
        self,
        nodes_file_path,
        output_edges_path,
    ):
        nodes = pd.read_csv(
            nodes_file_path
        )
        nodes.columns = [
            str(column).strip()
            for column in nodes.columns
        ]

        required_columns = {
            "x",
            "y",
            "w",
            "h",
            "g_ratio",
            "p_ratio",
            "isV",
        }
        missing_columns = (
            required_columns.difference(
                nodes.columns
            )
        )

        if missing_columns:
            raise ValueError(
                f"{nodes_file_path} is missing "
                f"columns: "
                f"{sorted(missing_columns)}"
            )

        edge_rows = []

        frames = nodes.groupby(
            ["g_ratio", "p_ratio"],
            sort=True,
        )

        for (
            global_progress,
            phase_progress,
        ), frame_nodes in frames:
            frame_nodes = (
                frame_nodes
                .reset_index(drop=True)
            )

            physical_nodes = (
                frame_nodes[
                    frame_nodes["isV"]
                    .astype(int)
                    == 0
                ]
            )
            virtual_nodes = (
                frame_nodes[
                    frame_nodes["isV"]
                    .astype(int)
                    == 1
                ]
            )

            if len(virtual_nodes) != 1:
                raise RuntimeError(
                    "Expected exactly one "
                    "virtual-context node in "
                    f"frame ({global_progress}, "
                    f"{phase_progress}); found "
                    f"{len(virtual_nodes)}"
                )

            positions = (
                physical_nodes[
                    ["x", "y"]
                ].to_numpy(
                    dtype=float
                )
            )
            dimensions = (
                physical_nodes[
                    ["w", "h"]
                ].to_numpy(
                    dtype=float
                )
            )
            physical_indices = (
                physical_nodes
                .index
                .tolist()
            )

            for first_index in range(
                len(physical_indices)
            ):
                for second_index in range(
                    first_index + 1,
                    len(physical_indices),
                ):
                    distance = np.linalg.norm(
                        positions[first_index]
                        - positions[second_index]
                    )
                    diagonal_first = np.sqrt(
                        dimensions[
                            first_index,
                            0,
                        ] ** 2
                        + dimensions[
                            first_index,
                            1,
                        ] ** 2
                    )
                    diagonal_second = np.sqrt(
                        dimensions[
                            second_index,
                            0,
                        ] ** 2
                        + dimensions[
                            second_index,
                            1,
                        ] ** 2
                    )

                    if distance <= (
                        diagonal_first
                        + diagonal_second
                    ) * self.tau:
                        source_index = int(
                            physical_indices[
                                first_index
                            ]
                        )
                        target_index = int(
                            physical_indices[
                                second_index
                            ]
                        )

                        edge_rows.append(
                            [
                                global_progress,
                                phase_progress,
                                source_index,
                                target_index,
                                "physical",
                            ]
                        )
                        edge_rows.append(
                            [
                                global_progress,
                                phase_progress,
                                target_index,
                                source_index,
                                "physical",
                            ]
                        )

            virtual_index = int(
                virtual_nodes.index[0]
            )

            for physical_index in (
                physical_indices
            ):
                physical_index = int(
                    physical_index
                )

                edge_rows.append(
                    [
                        global_progress,
                        phase_progress,
                        virtual_index,
                        physical_index,
                        "virtual",
                    ]
                )
                edge_rows.append(
                    [
                        global_progress,
                        phase_progress,
                        physical_index,
                        virtual_index,
                        "virtual",
                    ]
                )

        output = pd.DataFrame(
            edge_rows,
            columns=[
                "g_ratio",
                "p_ratio",
                "src",
                "dst",
                "type",
            ],
        )
        output.to_csv(
            output_edges_path,
            index=False,
        )

        return len(output)

### Example call: edge generation

Replace `case_directory` with one constructed S-STKG
`Case_XX` directory.

In [ ]:
case_directory = "/path/to/AutoLaparo_STKG/Case_01"

edge_generator = SurgicalEdgeGenerator(
    optimal_tau=0.30
)
edge_generator.process_all_files(
    folder_path=case_directory
)

## 5. Edge-Feature Generation

Physical edges receive Euclidean centroid distance, object-scale-normalized
distance, and bounding-box intersection-over-union. Virtual-context edges are
structural links and receive neutral zero-valued geometric descriptors.

In [ ]:
class SurgicalEdgeFeatureExtractor:
    @staticmethod
    def calculate_iou(
        box_first,
        box_second,
    ):
        x1, y1, w1, h1 = box_first
        x2, y2, w2, h2 = box_second

        first_x_min = x1 - w1 / 2
        first_y_min = y1 - h1 / 2
        first_x_max = x1 + w1 / 2
        first_y_max = y1 + h1 / 2

        second_x_min = x2 - w2 / 2
        second_y_min = y2 - h2 / 2
        second_x_max = x2 + w2 / 2
        second_y_max = y2 + h2 / 2

        intersection_x_min = max(
            first_x_min,
            second_x_min,
        )
        intersection_y_min = max(
            first_y_min,
            second_y_min,
        )
        intersection_x_max = min(
            first_x_max,
            second_x_max,
        )
        intersection_y_max = min(
            first_y_max,
            second_y_max,
        )

        intersection_width = max(
            0.0,
            intersection_x_max
            - intersection_x_min,
        )
        intersection_height = max(
            0.0,
            intersection_y_max
            - intersection_y_min,
        )
        intersection_area = (
            intersection_width
            * intersection_height
        )

        union_area = (
            w1 * h1
            + w2 * h2
            - intersection_area
        )

        return (
            intersection_area
            / union_area
            if union_area > 0
            else 0.0
        )

    @staticmethod
    def frame_key(
        global_progress,
        phase_progress,
    ):
        return (
            round(
                float(global_progress),
                4,
            ),
            round(
                float(phase_progress),
                4,
            ),
        )

    def process_all_files(
        self,
        folder_path,
    ):
        folder_path = Path(
            folder_path
        )

        if not folder_path.is_dir():
            raise FileNotFoundError(
                f"S-STKG case directory "
                f"was not found: "
                f"{folder_path}"
            )

        node_files = sorted(
            folder_path.glob(
                "*_nodes.txt"
            )
        )

        if not node_files:
            raise RuntimeError(
                f"No node files were found "
                f"in {folder_path}"
            )

        total_features = 0

        for node_path in node_files:
            base_name = (
                node_path.name.replace(
                    "_nodes.txt",
                    "",
                )
            )
            edge_path = (
                folder_path
                / f"{base_name}_edges.txt"
            )
            feature_path = (
                folder_path
                / (
                    f"{base_name}"
                    "_edge_features.txt"
                )
            )

            if not edge_path.exists():
                raise FileNotFoundError(
                    f"Matching edge file was "
                    f"not found: {edge_path}"
                )

            feature_count = (
                self.extract_features(
                    nodes_file=node_path,
                    edges_file=edge_path,
                    output_file=(
                        feature_path
                    ),
                )
            )
            total_features += (
                feature_count
            )

            print(
                f"  {base_name} | "
                f"edge_features="
                f"{feature_count}"
            )

        print(
            "Total edge-feature rows: "
            f"{total_features}"
        )

    def extract_features(
        self,
        nodes_file,
        edges_file,
        output_file,
    ):
        nodes = pd.read_csv(
            nodes_file
        )
        edges = pd.read_csv(
            edges_file
        )

        nodes.columns = [
            str(column).strip()
            for column in nodes.columns
        ]
        edges.columns = [
            str(column).strip()
            for column in edges.columns
        ]

        required_node_columns = {
            "x",
            "y",
            "w",
            "h",
            "g_ratio",
            "p_ratio",
            "isV",
        }
        required_edge_columns = {
            "g_ratio",
            "p_ratio",
            "src",
            "dst",
        }

        missing_node_columns = (
            required_node_columns.difference(
                nodes.columns
            )
        )
        missing_edge_columns = (
            required_edge_columns.difference(
                edges.columns
            )
        )

        if missing_node_columns:
            raise ValueError(
                f"{nodes_file} is missing "
                f"columns: "
                f"{sorted(missing_node_columns)}"
            )

        if missing_edge_columns:
            raise ValueError(
                f"{edges_file} is missing "
                f"columns: "
                f"{sorted(missing_edge_columns)}"
            )

        frame_groups = {}

        for (
            global_progress,
            phase_progress,
        ), frame_nodes in nodes.groupby(
            ["g_ratio", "p_ratio"],
            sort=True,
        ):
            key = self.frame_key(
                global_progress,
                phase_progress,
            )
            frame_groups[key] = (
                frame_nodes
                .reset_index(drop=True)
            )

        feature_rows = []

        for _, edge in edges.iterrows():
            key = self.frame_key(
                edge["g_ratio"],
                edge["p_ratio"],
            )

            if key not in frame_groups:
                raise RuntimeError(
                    "No matching node frame "
                    f"for edge frame {key}"
                )

            frame_nodes = (
                frame_groups[key]
            )
            source_index = int(
                edge["src"]
            )
            target_index = int(
                edge["dst"]
            )

            if (
                source_index < 0
                or target_index < 0
                or source_index
                >= len(frame_nodes)
                or target_index
                >= len(frame_nodes)
            ):
                raise IndexError(
                    "Edge references an invalid "
                    f"local node index in frame "
                    f"{key}: src={source_index}, "
                    f"dst={target_index}, "
                    f"nodes={len(frame_nodes)}"
                )

            source_node = (
                frame_nodes.iloc[
                    source_index
                ]
            )
            target_node = (
                frame_nodes.iloc[
                    target_index
                ]
            )

            if (
                int(source_node["isV"]) == 1
                or int(target_node["isV"]) == 1
            ):
                distance = 0.0
                relative_distance = 0.0
                iou = 0.0
            else:
                source_position = np.array(
                    [
                        source_node["x"],
                        source_node["y"],
                    ],
                    dtype=float,
                )
                target_position = np.array(
                    [
                        target_node["x"],
                        target_node["y"],
                    ],
                    dtype=float,
                )

                distance = float(
                    np.linalg.norm(
                        source_position
                        - target_position
                    )
                )

                source_diagonal = float(
                    np.sqrt(
                        source_node["w"] ** 2
                        + source_node["h"] ** 2
                    )
                )
                target_diagonal = float(
                    np.sqrt(
                        target_node["w"] ** 2
                        + target_node["h"] ** 2
                    )
                )

                relative_distance = (
                    distance
                    / (
                        (
                            source_diagonal
                            + target_diagonal
                        )
                        / 2
                    )
                    if (
                        source_diagonal
                        + target_diagonal
                    ) > 0
                    else 0.0
                )

                iou = self.calculate_iou(
                    [
                        source_node["x"],
                        source_node["y"],
                        source_node["w"],
                        source_node["h"],
                    ],
                    [
                        target_node["x"],
                        target_node["y"],
                        target_node["w"],
                        target_node["h"],
                    ],
                )

            feature_rows.append(
                [
                    key[0],
                    key[1],
                    source_index,
                    target_index,
                    round(distance, 4),
                    round(
                        relative_distance,
                        4,
                    ),
                    round(iou, 4),
                ]
            )

        output = pd.DataFrame(
            feature_rows,
            columns=[
                "g_ratio",
                "p_ratio",
                "src",
                "dst",
                "dist",
                "rel_dist",
                "iou",
            ],
        )
        output.to_csv(
            output_file,
            index=False,
        )

        return len(output)

### Example call: edge-feature generation

Use the same S-STKG case directory processed during edge
generation.

In [ ]:
case_directory = "/path/to/AutoLaparo_STKG/Case_01"

feature_extractor = (
    SurgicalEdgeFeatureExtractor()
)
feature_extractor.process_all_files(
    folder_path=case_directory
)

## Output

For each processed case and phase-aligned fragment, the notebook produces:

```text
<output_root>/Case_XX/
├── ..._nodes.txt
├── ..._labels.txt
├── ..._edges.txt
└── ..._edge_features.txt
```

Physical node class IDs are written directly from the trained detector without
runtime remapping. The programmatic virtual-context node uses `class=8` and
`isV=1`. Dataset-native phase labels remain unchanged except for the explicit
P1–P8 source phase-label encoding used by the Cholec80–M2CAI notebooks.